## Data Profiling Summary — orders_raw

Purpose: understand the raw data and record findings only. No changes made, no decisions here.

| # | Step | What was checked | Finding |
|---|------|------------------|---------|
| 1 | Table size | number of rows | 9,268 rows |
| 2 | Duplicate rows | exact duplicate rows | ~183 duplicate rows exist |
| 3 | Missing / blank values | empty values per column | customer_id missing in 103 rows; category missing in 79 rows; all other columns complete |
| 4 | Order statuses | distinct status values | completed 8,764 · refunded 403 · test 101 |
| 5 | Categories | distinct category values | 6 categories present, plus 79 rows with no category |
| 6 | Currencies & countries | distinct values | currencies: EUR (7,340), RON (1,928); countries: RO, DE, HU, BG |
| 7 | Quantity | range & invalid values | ranges from -3 to 3; 167 rows are zero or negative |
| 8 | Price | range & invalid values | ranges from 0 to 999,999; 24 rows priced 0; 13 rows priced 999,999 |
| 9 | FX reference dates | range & validity | span 2026-08-23 → 2026-09-03; all valid dates |
| 10 | Unique identifier | which column(s) uniquely identify a row | no single column is unique; the unique key is order_id + sku |


**Profiling conclusions:**
The dataset has 9,268 order-line rows. 

It contains duplicate rows, some missing values (customer_id, category), a non-real "test" status, out-of-range quantities and a placeholder
price, and dates in the expected FX range. 

No single column is unique — rows are identified by
order_id + sku. 


## 1. Size of the table
How many rows and columns are we dealing with?

In [3]:
%%sql
-- Counts every row in the table so we know how much data we have
SELECT COUNT(*) AS total_rows FROM orders_raw

StatementMeta(, 283a349f-4a6a-4241-a10e-f291cafd8435, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

## 2. Are there duplicate rows?
If total rows is higher than unique rows, some rows are exact copies.

In [4]:
%%sql
-- Compares total rows vs. rows that are truly unique.
-- COUNT(DISTINCT ...all columns...) counts each unique combination once,
-- so total minus unique = how many rows are exact copies.
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT order_id, customer_id, customer_email, order_ts, status, channel,
                 sku, product_name, category, qty, unit_price, currency, country, fx_reference_date) AS unique_rows,
  COUNT(*) - COUNT(DISTINCT order_id, customer_id, customer_email, order_ts, status, channel,
                 sku, product_name, category, qty, unit_price, currency, country, fx_reference_date) AS duplicate_rows
FROM orders_raw

StatementMeta(, 283a349f-4a6a-4241-a10e-f291cafd8435, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

## 3. Missing or blank values (per column)
Counts both truly empty (NULL) and blank text ("") for every column at once.

In [5]:
%%sql
-- For each column, adds 1 every time the value is empty (NULL) or blank ("").
-- SUM(CASE WHEN ... THEN 1 ELSE 0 END) is just "count the rows that match this condition".
SELECT
  SUM(CASE WHEN order_id          IS NULL OR TRIM(order_id)          = '' THEN 1 ELSE 0 END) AS null_order_id,
  SUM(CASE WHEN customer_id       IS NULL OR TRIM(customer_id)       = '' THEN 1 ELSE 0 END) AS null_customer_id,
  SUM(CASE WHEN customer_email    IS NULL OR TRIM(customer_email)    = '' THEN 1 ELSE 0 END) AS null_email,
  SUM(CASE WHEN order_ts          IS NULL OR TRIM(order_ts)          = '' THEN 1 ELSE 0 END) AS null_order_ts,
  SUM(CASE WHEN status            IS NULL OR TRIM(status)            = '' THEN 1 ELSE 0 END) AS null_status,
  SUM(CASE WHEN category          IS NULL OR TRIM(category)          = '' THEN 1 ELSE 0 END) AS null_category,
  SUM(CASE WHEN qty               IS NULL OR TRIM(qty)               = '' THEN 1 ELSE 0 END) AS null_qty,
  SUM(CASE WHEN unit_price        IS NULL OR TRIM(unit_price)        = '' THEN 1 ELSE 0 END) AS null_unit_price,
  SUM(CASE WHEN currency          IS NULL OR TRIM(currency)          = '' THEN 1 ELSE 0 END) AS null_currency,
  SUM(CASE WHEN country           IS NULL OR TRIM(country)           = '' THEN 1 ELSE 0 END) AS null_country,
  SUM(CASE WHEN fx_reference_date IS NULL OR TRIM(fx_reference_date) = '' THEN 1 ELSE 0 END) AS null_fx_date
FROM orders_raw

StatementMeta(, 283a349f-4a6a-4241-a10e-f291cafd8435, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 11 fields>

## 4.  What order statuses exist?

In [1]:
%%sql
-- Groups the rows by status and counts each one,
-- so we can spot statuses that aren't real sales (like 'test').
SELECT status, COUNT(*) AS how_many
FROM orders_raw
GROUP BY status
ORDER BY how_many DESC

StatementMeta(, 90cbd2c9-fb8c-483b-a6f9-1facbaa23814, 2, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

## 5. What categories exist?

In [2]:
%%sql
-- Lists each category with its row count; a NULL group reveals the missing categories.
SELECT category, COUNT(*) AS how_many
FROM orders_raw
GROUP BY category
ORDER BY how_many DESC

StatementMeta(, 90cbd2c9-fb8c-483b-a6f9-1facbaa23814, 3, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 2 fields>

## 6. Currencies and countries

In [3]:
%%sql
-- We need to know which currencies must be converted to EUR later.
-- Lists currencies and countries in one result.
-- UNION ALL just stacks the two lists on top of each other.
SELECT 'currency' AS field, currency AS value, COUNT(*) AS how_many FROM orders_raw GROUP BY currency
UNION ALL
SELECT 'country' AS field, country AS value, COUNT(*) AS how_many FROM orders_raw GROUP BY country
ORDER BY field, how_many DESC

StatementMeta(, 90cbd2c9-fb8c-483b-a6f9-1facbaa23814, 4, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 3 fields>

## 7. Quantity checks

In [4]:
%%sql
-- Look for zero/negative quantities and any valu that isn't a whole number.
-- Checks quantities: smallest/largest value, how many are zero-or-negative,
-- and how many aren't a proper whole number. RLIKE tests the text against a number pattern.
SELECT
  MIN(CAST(qty AS INT)) AS min_qty,
  MAX(CAST(qty AS INT)) AS max_qty,
  SUM(CASE WHEN CAST(qty AS INT) <= 0 THEN 1 ELSE 0 END) AS zero_or_negative,
  SUM(CASE WHEN qty IS NULL OR qty NOT RLIKE '^-?[0-9]+$' THEN 1 ELSE 0 END) AS not_a_whole_number
FROM orders_raw

StatementMeta(, 90cbd2c9-fb8c-483b-a6f9-1facbaa23814, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

## 8. Price checks

In [5]:
%%sql
-- Look for zero prices, the fake 999999 placeholder, and any non-numeric price.
-- Checks prices: smallest/largest, how many are 0, how many are the fake 999999 placeholder,
-- and how many aren't a valid number at all.
SELECT
  MIN(CAST(unit_price AS DOUBLE)) AS min_price,
  MAX(CAST(unit_price AS DOUBLE)) AS max_price,
  SUM(CASE WHEN CAST(unit_price AS DOUBLE) = 0 THEN 1 ELSE 0 END) AS zero_price,
  SUM(CASE WHEN CAST(unit_price AS DOUBLE) = 999999 THEN 1 ELSE 0 END) AS placeholder_999999,
  SUM(CASE WHEN unit_price IS NULL OR unit_price NOT RLIKE '^[0-9]+([.][0-9]+)?$' THEN 1 ELSE 0 END) AS not_a_number
FROM orders_raw

StatementMeta(, 90cbd2c9-fb8c-483b-a6f9-1facbaa23814, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 5 fields>

## 9. FX reference dates

In [6]:
%%sql
-- These decide which exchange rate to use. Check the range and that all are valid dates.
-- Finds the earliest and latest FX reference date, and counts any that aren't valid dates.
-- TRY_CAST turns bad dates into NULL instead of erroring.
SELECT
  MIN(TRY_CAST(fx_reference_date AS DATE)) AS earliest_fx_date,
  MAX(TRY_CAST(fx_reference_date AS DATE)) AS latest_fx_date,
  SUM(CASE WHEN TRY_CAST(fx_reference_date AS DATE) IS NULL THEN 1 ELSE 0 END) AS invalid_dates
FROM orders_raw

StatementMeta(, 90cbd2c9-fb8c-483b-a6f9-1facbaa23814, 7, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

## 10. What is the unique identifier?

In [2]:
%%sql
-- Find out if any single column can uniquely identify a row.
-- Count the total rows, then count the distinct (different) values in each column.
-- If a column's distinct count equals total_rows, that column is unique.
-- if all of them are smaller, then no single column is a unique identifier.

SELECT
  COUNT(*)                     AS total_rows,
  COUNT(DISTINCT order_id)     AS distinct_order_id,
  COUNT(DISTINCT sku)          AS distinct_sku,
  COUNT(DISTINCT customer_id)  AS distinct_customer_id,
  COUNT(DISTINCT customer_email) AS distinct_customer_email
FROM orders_raw

StatementMeta(, 603c63ce-25ab-4483-b5b7-47cdd8c76cf1, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 5 fields>